# 03 — Build your own corpus (PDFs + Markdown + text)

Upload a few PDFs or Markdown files, build a `CorpusIndex` from them, and query. Then attach the index to the engine so the research agent reasons over *your* documents.

**You will learn:**
1. Build a `CorpusIndex` from a directory using `scripts/index_corpus.py` (and via the Python API).
2. Save the index to Google Drive so it survives Colab disconnects.
3. Query the index directly for fast lookup (no LLM needed).
4. Hook the index into the engine via `LOCAL_CORPUS_PATH` + `personal_docs` preset.

**Prerequisites:** 3–5 PDFs on your laptop to upload. Optional: Groq API key from Tutorial 02 for the end-to-end step. **Time:** ~5 minutes. **Cost:** $0.

## Step 1 — install

In [ ]:
%%capture
!git clone --depth 1 https://github.com/TheAiSingularity/agentic-research-engine-oss.git /content/engine-repo
!pip install -q rank_bm25 pypdf trafilatura openai

import sys, os
sys.path.insert(0, '/content/engine-repo')

# We need an embedder to build the index. Use a free-tier Groq embedding?
# Actually Groq doesn't expose embeddings. Use sentence-transformers locally
# for fully offline indexing (tiny model, ~100MB).
!pip install -q sentence-transformers

In [ ]:
# Monkey-patch core.rag's embedder to use a local sentence-transformers model
# instead of calling any API. This keeps Tutorial 03 fully offline.
from sentence_transformers import SentenceTransformer

_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def local_embed(batch):
    return _model.encode(list(batch), show_progress_bar=False).tolist()

import core.rag.python.rag as _rag_mod
_rag_mod._openai_embedder = local_embed

import core.rag.python.hybrid as _hybrid_mod
_hybrid_mod._openai_embedder = local_embed

print('embedder:', local_embed('hello'.split())[0][:5], '...')

## Step 2 — upload files

Click the 📁 icon in the left sidebar → upload. Or run the cell below which opens Colab's file picker.

In [ ]:
import os, pathlib

CORPUS_DIR = pathlib.Path('/content/my-docs')
CORPUS_DIR.mkdir(exist_ok=True)

# If you want the Colab file picker, uncomment:
# from google.colab import files
# for fname, data in files.upload().items():
#     (CORPUS_DIR / fname).write_bytes(data)

# For a reproducible demo, write a couple of sample markdown files.
(CORPUS_DIR / 'rag-notes.md').write_text('''# RAG Notes

Hybrid retrieval combines BM25 (sparse, token-level) with dense embeddings using RRF with k=60.

Cross-encoder reranking re-scores (query, passage) pairs with attention. BAAI/bge-reranker-v2-m3 is a strong open-source choice.

Anthropic Contextual Retrieval prepends each chunk with an LLM-generated one-sentence summary of where it fits in its source document. They report a 49% reduction in retrieval failures over plain chunking, rising to 67% when combined with reranking.
''')

(CORPUS_DIR / 'agents-notes.md').write_text('''# Agent notes

CoVe (Chain-of-Verification, Dhuliawala et al. 2023) decomposes an answer into atomic claims and checks each against the evidence pool.

FLARE (Jiang et al. 2023) detects hedged phrases in a draft answer and re-searches for the specific claim.

MiroThinker-H1 (2026) scores 88.2 on BrowseComp using 300+ tool interactions per query.
''')

print('Files in corpus:')
for p in CORPUS_DIR.iterdir():
    print('  ', p.name, f'{p.stat().st_size:>6d} bytes')

## Step 3 — build the index

Two ways: via the CLI script (same command your users run on their machines), or via the Python API (handier in a notebook).

In [ ]:
# Python API — directly use core.rag.CorpusIndex
from core.rag import CorpusIndex

idx = CorpusIndex.build(CORPUS_DIR, chunk_size=600, overlap=100)
print(f'Built index: {len(idx.chunks)} chunks from {len({c.source for c in idx.chunks})} sources')

INDEX_DIR = pathlib.Path('/content/my-docs.idx')
idx.save(INDEX_DIR)
print(f'Saved to {INDEX_DIR}')

In [ ]:
# Inspect the manifest (it's human-readable JSON).
!cat /content/my-docs.idx/manifest.json

## Step 4 — query the corpus directly

`CorpusIndex.query(question, k=5)` returns the top-k `(CorpusChunk, cosine_score)` tuples. Fast lookup, no LLM needed.

In [ ]:
loaded = CorpusIndex.load(INDEX_DIR)
hits = loaded.query('contextual retrieval failure rate', k=3)
for chunk, score in hits:
    loc = f"corpus://{chunk.source}" + (f"#p{chunk.page}" if chunk.page is not None else '') + f"#c{chunk.chunk_idx}"
    print(f'[{score:.3f}] {loc}')
    print('  preview:', chunk.text[:140].replace('\n', ' '))
    print()

## Step 5 — (optional) save to Google Drive

Colab wipes `/content` on disconnect. Copy the index to Drive so your next session resumes instantly.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree('/content/my-docs.idx', '/content/drive/MyDrive/my-docs.idx', dirs_exist_ok=True)
# print('Index saved to Google Drive at /content/drive/MyDrive/my-docs.idx')

print('(commented out — uncomment + approve the Drive auth prompt to run)')

## Step 6 — hook it into the engine

Set `LOCAL_CORPUS_PATH` and pick the `personal_docs` domain preset. The engine will merge corpus hits into evidence and `_fetch_url` will skip them (they're already full chunks).

For the end-to-end step we need an LLM. Either:
- (A) Plug in Groq as Tutorial 02 showed, OR
- (B) Use the mocked LLM from Tutorial 01 — shown here for reproducibility.

In [ ]:
# Path B: mocked LLM so this cell always runs without API keys.
from types import SimpleNamespace
from unittest import mock
import engine.core.models as _models

def _resp(text):
    return SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=text))])

def canned(*args, **kwargs):
    p = kwargs.get('messages', [{}])[0].get('content', '')
    if 'Classify' in p:                      return _resp('multihop')
    if 'step-level verifier' in p:            return _resp('VERDICT: accept\nFEEDBACK:')
    if 'Break this research question' in p:   return _resp('contextual retrieval\nreranking\ncombined effect')
    if 'Compress each numbered chunk' in p:   return _resp('[1] compressed A\n[2] compressed B\n[3] compressed C')
    if 'Answer the question using ONLY' in p: return _resp('Contextual Retrieval reduces retrieval failures by 49% alone, 67% combined with reranking [1][2].')
    if 'List each standalone factual' in p:   return _resp('CLAIM: CR reduces failures 49% alone\nVERIFIED: yes\nCLAIM: combined with reranking reaches 67%\nVERIFIED: yes')
    if 'Summarize these sources' in p:         return _resp('Summary [1] [2].')
    return _resp('(stub)')

cli = mock.MagicMock(); cli.chat.completions.create.side_effect = canned
_models.OpenAI = mock.MagicMock(return_value=cli)

# Point the engine at our local index, route to personal_docs preset.
os.environ['LOCAL_CORPUS_PATH'] = str(INDEX_DIR)
os.environ['ENABLE_FETCH'] = '0'
os.environ['ENABLE_STREAM'] = '0'

from engine.interfaces.common import run_query, format_sources, format_verified_summary

result = run_query(
    'What does my corpus say about contextual retrieval and reranking?',
    domain='personal_docs',
)
print('Answer:', result.answer)
print()
print(format_verified_summary(result))
for row in format_sources(result):
    print(f"  [{row['idx']}] {row['url']}")

## What to try next

- Upload your actual papers / notes / reports via the file picker.
- Tune `chunk_size` + `overlap` in `CorpusIndex.build` — smaller chunks retrieve more precisely, larger chunks preserve more context.
- Combine with Tutorial 02's Groq setup for a real-LLM end-to-end over your docs.
- Run the CLI version locally: `python scripts/index_corpus.py build ~/papers --out ~/papers.idx`.

Next tutorial: **04 — Drive the engine from a Python MCP client**.